The following notebook creates a Data Table in Lh_datalake. 
Second, based on this table a view in lh_enriched will be created to send to the Semantic model and dimension table in datawarehouse on a later moment.

This notebook has only to be executed once.

In [ ]:
!pip install babel

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, weekofyear, last_day, trunc
from datetime import datetime, timedelta
import pandas as pd
from babel.dates import format_date

def quarter(date):
    return (date.month - 1) // 3 + 1

def fill_d_datum_spark(startyear: str, endyear: str) -> pd.DataFrame:
    """creation of a date table in the lakehouse, based on the start and endyear scope"""

    #dutch
    language = 'nl_NL'
    #english
    #language = 'en_US'

    startyear = startyear
    endyear = endyear
    day_date = datetime.strptime(f'{startyear}0101', '%Y%m%d')
    endday_date = datetime.strptime(f'{endyear + 1}0101', '%Y%m%d')

    data = {'datum_id': [], 'datum': [],'datum_datum': [], 'cal_year': [], 'cal_quarter': [],
            'cal_quarter_year_description': [], 'cal_quarter_code': [],'cal_quarter_description': [],'cal_month': [], 'cal_month_short_description': [],
            'cal_month_long_description': [], 'cal_month_code':[], 'cal_month_code_description':[],'cal_day_of_the_year': [], 'cal_day_of_the_month': [], 'day_of_the_week': [],
            'weekday_description': [], 'weekday_short_description':[],'is_weekend': [], 'is_holiday': [], 'holiday_description': [],
            'period_id_week': [], 'period_id_month': [], 'period_id_quarter': [], 'period_id_year': [], 'week': [],'iso_year': [],
            'iso_week': [], 'iso_yearweek': [], 'iso_yearweek_description': [],
            'start_date': [], 'end_date': []}

    while day_date < endday_date:
        X = day_date.year

        K = X // 100
        M = 15 + (3 * K + 3) // 4 - (8 * K + 13) // 25
        S = 2 - (3 * K + 3) // 4
        A = X % 19
        D = (19 * A + M) % 30
        R = (D + A // 11) // 29
        OG = 21 + D - R
        SZ = 7 - (X + X // 4 + S) % 7
        OE = 7 - (OG - SZ) % 7
        OS = OG + OE
        EASTERDATE = (datetime(X, 3, 1) + timedelta(days=OS - 1)).date()
        day_id = day_date.strftime('%Y%m%d')
        weekday = day_date.weekday() + 1 if day_date.weekday() != 6 else 7

        feastday_description = None
        if day_date.strftime('%m%d') == '1225':
            feastday_description = '1e Kerstdag'
        elif day_date.strftime('%m%d') == '1226':
            feastday_description = '2e Kerstdag'
        elif day_date.strftime('%m%d') == '0101':
            feastday_description = 'Nieuwjaarsdag'
        elif day_date.date() == EASTERDATE:
            feastday_description = '1e Paasdag'
        elif day_date.date() == EASTERDATE + timedelta(days=1):
            feastday_description = '2e Paasdag'
        elif day_date.date() == EASTERDATE + timedelta(days=39):
            feastday_description = 'Hemelvaartsdag'
        elif day_date.date() == EASTERDATE + timedelta(days=40) and day_date.year == 2021:
            feastday_description = 'Hemelvaart_extradag'
        elif day_date.date() == EASTERDATE + timedelta(days=49):
            feastday_description = '1e Pinksterdag'
        elif day_date.date() == EASTERDATE + timedelta(days=50):
            feastday_description = '2e Pinksterdag'
        elif day_date.year >= 2014 and day_date.strftime('%m%d') == '0427' and weekday != 7:
            feastday_description = 'Koningsdag'
        elif day_date.year >= 2014 and day_date.strftime('%m%d') == '0426' and weekday == 6:
            feastday_description = 'Koningsdag'
        elif 1980 <= day_date.year <= 2013 and day_date.strftime('%m%d') == '0430' and weekday != 7:
            feastday_description = 'Koninginnedag'
        elif 1980 <= day_date.year <= 2013 and day_date.strftime('%m%d') == '0429' and weekday == 6:
            feastday_description = 'Koninginnedag'
        elif 1948 <= day_date.year <= 1979 and day_date.strftime('%m%d') == '0430' and weekday != 7:
            feastday_description = 'Koninginnedag'
        elif 1948 <= day_date.year <= 1979 and day_date.strftime('%m%d') == '0501' and weekday == 1:
            feastday_description = 'Koninginnedag'

        data['datum_id'].append(int(day_id))
        data['datum'].append(day_date.strftime("%Y-%m-%d %H:%M:%S"))
        data['datum_datum'].append(day_date.strftime('%Y-%m-%d'))
        data['cal_year'].append(day_date.year)
        data['cal_quarter'].append(quarter(day_date))
        data['cal_quarter_year_description'].append(f"{day_date.year} - Q{quarter(day_date)}")
        data['cal_quarter_code'].append(f"Q{quarter(day_date)}")
        data['cal_quarter_description'].append(f"Kwartaal {quarter(day_date)}")

        data['cal_month'].append(day_date.month)
        data['cal_month_code'].append(day_date.strftime('%m'))
        data['cal_month_code_description'].append(f"M{day_date.strftime('%m')}")
        data['cal_month_short_description'].append(format_date(day_date,'MMM', locale = language).capitalize())

        data['cal_month_long_description'].append(format_date(day_date,'MMMM', locale = language).capitalize())
        data['cal_day_of_the_month'].append(int(day_date.strftime('%d')))
        data['cal_day_of_the_year'].append(int(day_date.strftime('%j')))
        data['day_of_the_week'].append(weekday)

        data['weekday_description'].append(format_date(day_date,'EEEE', locale = language).capitalize())
        data['weekday_short_description'].append(format_date(day_date,'EEE', locale = language).capitalize())
        data['is_weekend'].append(1 if weekday in (6, 7) else 0)
        data['is_holiday'].append(1 if feastday_description else 0)
        data['holiday_description'].append(feastday_description)
        data['week'].append(int(day_date.strftime('%W')))
        
        data['period_id_week'].append(int(f"{day_date.year}{day_date.strftime('%W')}"))
        data['period_id_month'].append(int(day_date.strftime('%Y%m')))
        data['period_id_quarter'].append(int(f"{day_date.year}{quarter(day_date)}"))
        data['period_id_year'].append(day_date.year)
   
        data['iso_year'].append(int(day_date.strftime('%G')))
        data['iso_week'].append(int(day_date.strftime('%V')))
        data['iso_yearweek'].append(int(f"{day_date.strftime('%G')}0000{day_date.strftime('%V')}"))
        data['iso_yearweek_description'].append(f"{day_date.strftime('%G')} - WK{day_date.strftime('%V').zfill(2)}")
        data['start_date'].append(day_date.strftime('%Y-%m-%d'))
        data['end_date'].append((day_date + timedelta(days=1) - timedelta(seconds=1)).strftime('%Y-%m-%d'))

        day_date += timedelta(days=1)

    # Create a PySpark DataFrame
    spark_df = spark.createDataFrame(pd.DataFrame(data))
    spark_df = spark_df.withColumn("first_day_of_month", trunc("datum", "month"))
    spark_df = spark_df.withColumn("last_day_of_month", last_day("datum"))

    return spark_df

In [ ]:
start = 2000
eind = 2040

spark_df = fill_d_datum_spark(start,eind)
spark_df.write.format('delta').mode("overwrite").saveAsTable('lh_datalake.DBO_date')


In [ ]:
%%sql
CREATE OR REPLACE view lh_enriched.vw_dim_date
as
SELECT

CAST(datum_id AS BIGINT) AS date_KEY,  
cast(datum_datum as DATE) as Datum, 
cast(datum_datum as varchar(20)) as DatumString, 
to_timestamp(datum) AS DatumTijd,
cal_year as Jaar,
CONCAT('Year ', cal_year) AS JaarOmschrijving,

    CASE 
        WHEN MONTH(datum) <= 4 THEN 1 
        WHEN MONTH(datum) <= 8 THEN 2 
        ELSE 3 
    END AS Tertiaal,
    cal_Quarter as Kwartaal,
    cal_quarter_code AS KwartaalCode,
    cal_quarter_description AS KwartaalOmschrijving,

cal_month as Maand, 
cal_month_long_description as MaandNaam,
cal_month_code_description as MaandCode,
cal_month_short_description as MaandNaamKort,

first_day_of_month AS EersteDagMaand,
last_day_of_month AS LaatsteDagMaand,

period_id_month AS JaarMaandOrder, 
period_id_week as JaarWeekOrder,
period_id_quarter as JaarKwartaalOrder,

case when cal_month in (1,4,7,10) then 1
when cal_month in (2,5,8,11) then 2
else 3 end as MaandInKwartaalOrder,

week as Week,

iso_week as ISOWeek,
iso_year as ISOJaar,

day_of_the_week as DagVanDeWeek,
cal_day_of_the_month as DagVanDeMaand,
weekday_description as WeekdagOmschrijving,
cal_day_of_the_year AS DagVanHetJaar,


is_holiday as IsFeestdag,
Holiday_description as FeestdagOmschrijving,

CASE WHEN datum = first_day_of_month THEN 'Ja' ELSE 'Nee' END AS EersteDagMaandInd,
CASE WHEN datum = last_day_of_month THEN 'Ja' ELSE 'Nee' END AS LaatsteDagMaandInd,
is_weekend as IsWeekend,
CASE WHEN is_weekend=0 then 1 else 0 end as WerkdagInd,

CASE WHEN is_weekend=0 then 8 else 0 end as Werkuren,

-- logic for reporting purposes
-- filtert elke dag gisteren 
CASE WHEN cast(datum_datum as DATE) = date_sub(current_date(), 1) then True 
else False end as Gisteren,

-- filtert elke dag eergisteren 
CASE WHEN cast(datum_datum as DATE) = date_sub(current_date(), 2) then True 
else False end as Eergisteren,

-- filtert alle dagen voor vandaag 
CASE WHEN cast(datum_datum as DATE) <= date_sub(current_date(), 1) then True 
else False end as Afgesloten,

-- filtert alle dagen maximaal een week geleden 
CASE WHEN cast(datum_datum as DATE) between date_sub(current_date(), 7) and date_sub(current_date(), 1) then True 
else False end as Binnen1Week,

-- filtert alle dagen maximaal een maand geleden 
CASE WHEN month(cast(datum_datum as DATE)) = month(cast(current_date() as date)) and day(cast(datum_datum as DATE)) < day(cast(current_date()as date)) and cal_year = year(current_date()) then True 
    WHEN month(cast(datum_datum as DATE)) = month(add_months(cast(current_date() as date),-1)) and day(cast(datum_datum as DATE)) >= day(cast(current_date() as date)) 
    and  (cal_year = year(current_date())
        or (cal_year = year(current_date())-1 and month(add_months(cast(current_date() as date),-1)) =12 ) 
        )
     then True 

    else False 

end as Binnen1Maand,


-- filtert alle dagen maximaal 2 maanden geleden 
CASE WHEN month(cast(datum_datum as DATE)) = month(cast(current_date() as date)) and day(cast(datum_datum as DATE)) < day(cast(current_date()as date)) and cal_year = year(current_date()) then True 
    WHEN month(cast(datum_datum as DATE)) = month(add_months(cast(current_date() as date),-1)) 
    and  (cal_year = year(current_date())
        or (cal_year = year(current_date())-1 and month(add_months(cast(current_date() as date),-1)) =12 ) 
        )
     then True 
    
    WHEN month(cast(datum_datum as DATE)) = month(add_months(cast(current_date() as date),-2)) and day(cast(datum_datum as DATE)) >= day(cast(current_date() as date)) 
    and  (cal_year = year(current_date())
        or (cal_year = year(current_date())-1 and month(add_months(cast(current_date() as date),-2)) =12 ) 
        )
     then True 

    else False 

end as Binnen2Maanden,

-- filtert alle dagen maximaal een jaar / 12 maanden geleden 
case when cast(datum_datum as DATE) <= date_sub(current_date(), 1) and cast(datum_datum as DATE) > cast(add_months(current_date(), -12) as DATE) then True 
    else False
end as Binnen12Maanden 

FROM
Lh_datalake.DBO_Date
order by Datum asc


In [1]:
%%sql

CREATE OR REPLACE view lh_enriched.vw_dim_boekjaar
as
select 
CAST
(cal_year AS BIGINT) AS Boekjaar_KEY,  

cal_year as Jaar,
CONCAT('Year ', cal_year) AS JaarOmschrijving

from Lh_datalake.DBO_Date

StatementMeta(, b8222f65-d08c-435c-a8d2-cc5abed05152, 2, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [2]:
df = spark.sql("SELECT * FROM lh_enriched.vw_dim_boekjaar")
df.write.format('delta').mode("overwrite").saveAsTable('lh_datawarehouse.dim_boekjaar')

StatementMeta(, b8222f65-d08c-435c-a8d2-cc5abed05152, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 022a8118-61f8-4a65-b6c8-96e7f950c096)